# Text Ingestion for RAG

This notebook covers production-style ingestion for text files:
- Robust project/data path resolution
- Single file and folder ingestion
- Metadata-first previews
- Chunking for downstream retrieval

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


PROJECT_ROOT = Path.cwd().resolve().parents[1]
DATA_DIR = PROJECT_ROOT / "data"


def get_data_file(file_name: str, base_dir: Path = DATA_DIR) -> Path:
    matches = list(base_dir.glob(f"**/{file_name}"))
    if not matches:
        raise FileNotFoundError(f"'{file_name}' was not found under {base_dir}")
    return matches[0]


print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)

In [ ]:
target_file = get_data_file("sample_dataset.txt")

single_loader = TextLoader(str(target_file), encoding="utf-8")
single_docs = single_loader.load()

print("Single-file ingestion")
print("documents:", len(single_docs))
print("source:", single_docs[0].metadata.get("source"))
print("preview:", single_docs[0].page_content[:140], "...")

In [ ]:
folder_loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
folder_docs = folder_loader.load()

print("\nFolder ingestion")
print("text documents loaded:", len(folder_docs))
print("sample sources:")
for d in folder_docs[:3]:
    print("-", d.metadata.get("source"))

In [ ]:
CHUNK_SIZE = 700
CHUNK_OVERLAP = 100

if CHUNK_SIZE <= 0:
    raise ValueError("CHUNK_SIZE must be > 0")
if CHUNK_OVERLAP < 0 or CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError("CHUNK_OVERLAP must be >= 0 and < CHUNK_SIZE")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = splitter.split_documents(single_docs)

print("\nChunking")
print("chunks:", len(chunks))
print("first chunk chars:", len(chunks[0].page_content))
print("first chunk metadata:", chunks[0].metadata)